# pandas 之八：完整案例分析（企鹅数据）

---

前面七篇都是"零件教学"，这一篇把零件**组装成一台机器** ——
从一个原始 CSV 文件出发，走完整个数据分析流程。

## 数据分析的标准五步

```
① 导入库和数据
② 数据探查（看长什么样、有没有脏）
③ 数据清洗（处理缺失值/重复值/类型）
④ 特征构造（造出更有分析价值的新列）
⑤ 分组分析 + 得出结论
```

**这套流程在任何数据集上都通用**，学会一次就能反复用。

## 数据集说明

用的是 Palmer 企鹅数据集（R 语言社区里的"iris 之后最强入门数据集"）：

| 列名 | 含义 |
|---|---|
| `species` | 企鹅种类（Adelie / Chinstrap / Gentoo） |
| `island` | 所在岛屿 |
| `bill_length_mm` | 喙长（毫米） |
| `bill_depth_mm` | 喙深（毫米） |
| `flipper_length_mm` | 鳍长（毫米） |
| `body_mass_weight_g` | 体重（克） |
| `sex` | 性别 |

**分析目标**：不同种类、不同岛屿、不同性别的企鹅，身体特征有什么差异？


In [ ]:
import pandas as pd
import numpy as np

print('pandas 版本：', pd.__version__)


---
# 第 ① 步：导入数据


In [ ]:
penguins = pd.read_csv('Penguins.csv')
print('读取成功！形状：', penguins.shape)


---
# 第 ② 步：数据探查

**⛔ 新手最容易犯的错：拿到数据直接开始算。**
✅ 正确做法：先花两分钟"认识"这份数据。


In [ ]:
print('=== 1) 前 5 行：肉眼看看数据长什么样 ===')
print(penguins.head())


In [ ]:
print('=== 2) 结构信息：列名、类型、非空数量 ===')
penguins.info()


从 `info()` 能读出三个关键信息：

1. **一共 500 行**（`RangeIndex: 500 entries`）
2. **有几列有缺失**：`body_mass_weight_g` 是 476 non-null，说明**少了 24 个**
3. **类型是否正确**：`species`/`island`/`sex` 是 `object`（文字），其余是 `float64`


In [ ]:
print('=== 3) 数值列的统计概览 ===')
print(penguins.describe().round(2))


从 `describe()` 能初步判断**有没有异常值**：

| 列 | min | max | 判断 |
|---|---|---|---|
| `bill_length_mm` | ~30 | ~60 | 正常 |
| `body_mass_weight_g` | 看是否出现负数或 0 | 正常 |

**如果 min 出现负数（比如体重 -100），那一定有问题。**


In [ ]:
print('=== 4) 分类列的取值分布 ===')
for col in ['species', 'island', 'sex']:
    print(f'\n{col}:')
    print(penguins[col].value_counts(dropna=False))


> **注意 `value_counts(dropna=False)`**：
> 默认 `value_counts()` **不统计缺失值**，
> 加上 `dropna=False` 才能看到"有多少个缺失"。


In [ ]:
print('=== 5) 专门统计缺失值 ===')
print(penguins.isnull().sum())
print('\n缺失比例：')
print((penguins.isnull().sum() / len(penguins) * 100).round(1))


**诊断结论**：

| 列 | 缺失数 | 处理方案 |
|---|---|---|
| `sex` | 少量 | 占比小，可以删 |
| `bill_*_mm` / `flipper_length_mm` | 少量 | 占比小，可以删 |
| `body_mass_weight_g` | 24 个 | 占比约 5%，**删掉影响不大** |

**决策：缺失比例都很低（<5%），直接删掉含缺失的行最简单可靠。**


---
# 第 ③ 步：数据清洗

### 处理策略怎么选？先看"缺失比例"

| 缺失比例 | 推荐策略 |
|---|---|
| < 5% | **直接删行**（影响小） |
| 5% ~ 30% | **填充**（均值/中位数/众数/前后值） |
| > 30% | 考虑**整列删掉**（信息太少，填了反而失真） |

我们这里缺失比例很低，采取"删行"。


In [ ]:
before = len(penguins)
penguins = penguins.dropna()
after = len(penguins)

print(f'清洗前：{before} 行')
print(f'清洗后：{after} 行')
print(f'删除了 {before - after} 行（占 {(before - after) / before * 100:.1f}%）')
print('\n再检查一遍，应该全是 0：')
print(penguins.isnull().sum().sum())


### 顺便检查重复值


In [ ]:
dup_count = penguins.duplicated().sum()
print('完全重复的行数：', dup_count)
if dup_count > 0:
    penguins = penguins.drop_duplicates()
    print('已去重，现在行数：', len(penguins))
else:
    print('没有重复行，不用处理')


### 规范数据类型

`species` / `island` / `sex` 是三列"只有少数几个取值"的文本，
转成 `category` 类型更省内存，也让后续分组更快。


In [ ]:
print('转换前 dtypes：')
print(penguins[['species', 'island', 'sex']].dtypes)

for col in ['species', 'island', 'sex']:
    penguins[col] = penguins[col].astype('category')

print('\n转换后 dtypes：')
print(penguins[['species', 'island', 'sex']].dtypes)


---
# 第 ④ 步：特征构造

**"特征构造"是数据分析里最能体现水平的一步** ——
原始列往往不够用，需要根据业务理解造出更有信息量的新列。

### 特征 1：喙的"形状比例"

喙长和喙深单独看，区分度有限。但它们的**比例**能刻画"喙是细长还是宽短"，
这对区分企鹅种类特别有效。


In [ ]:
penguins['bill_ratio'] = penguins['bill_length_mm'] / penguins['bill_depth_mm']
penguins['bill_ratio'] = penguins['bill_ratio'].round(2)

print('新增 bill_ratio（喙长/喙深）：')
print(penguins[['species', 'bill_length_mm', 'bill_depth_mm', 'bill_ratio']].head())


### 特征 2：体重分档

连续数值不便于"分组对比"，切成 `low / medium / high` 三档后，
可以和种类、岛屿做交叉分析。


In [ ]:
labels = ['low', 'medium', 'high']
penguins['body_category'] = pd.cut(penguins['body_mass_weight_g'], bins=3, labels=labels)

print('体重分档结果：')
print(penguins['body_category'].value_counts().sort_index())


In [ ]:
print('\n看看这三档对应的体重区间：')
print(penguins.groupby('body_category', observed=True)['body_mass_weight_g'].agg(['min', 'max', 'count']))


> **`observed=True` 是什么？**
> 用 category 列分组时，pandas 默认会把**所有出现过的类别**都列出来
> （哪怕这一组为空）。加 `observed=True` 只显示实际存在的组，结果更干净。


---
# 第 ⑤ 步：分组分析


## 分析 1：三种企鹅的身体特征差异


In [ ]:
by_species = penguins.groupby('species', observed=True).agg(
    数量=('species', 'size'),
    平均喙长=('bill_length_mm', 'mean'),
    平均喙深=('bill_depth_mm', 'mean'),
    平均鳍长=('flipper_length_mm', 'mean'),
    平均体重=('body_mass_weight_g', 'mean'),
    平均喙比例=('bill_ratio', 'mean'),
).round(2).sort_values('平均体重', ascending=False)

print('=' * 80)
print('三种企鹅的特征对比')
print('=' * 80)
print(by_species)


**初步结论**：

- **Gentoo** 是最"壮"的：体重最大、鳍最长
- **Adelie** 和 **Chinstrap** 体型接近，但 Chinstrap 喙更长
- `bill_ratio` 确实是个有区分度的特征


## 分析 2：按岛屿 + 性别分组（多列分组）


In [ ]:
by_island_sex = penguins.groupby(['island', 'sex'], observed=True).agg(
    数量=('species', 'size'),
    平均体重=('body_mass_weight_g', 'mean'),
    体重标准差=('body_mass_weight_g', 'std'),
    平均喙长=('bill_length_mm', 'mean'),
).round(2)

print('按「岛屿 + 性别」分组：')
print(by_island_sex)


**为什么要有"标准差"这一列？**
- **平均值**告诉你"水平如何"
- **标准差**告诉你"差异有多大"

如果某组标准差特别大，说明这组内部很"杂"，可能混了不同种类的企鹅。


## 分析 3：单变量分组（只按岛屿）


In [ ]:
by_island = penguins.groupby('island', observed=True).agg(
    数量=('species', 'size'),
    平均体重=('body_mass_weight_g', 'mean'),
    体重标准差=('body_mass_weight_g', 'std'),
).round(2)

print('按岛屿分组：')
print(by_island)


## 分析 4：交叉分析（种类 × 岛屿）


In [ ]:
cross = pd.crosstab(penguins['species'], penguins['island'])
print('种类 × 岛屿 的交叉表（每个组合有多少只）：')
print(cross)
print('\n  <- 一眼看出：Gentoo 只在 Biscoe 岛，Chinstrap 只在 Dream 岛')


**`pd.crosstab` 是做"两个分类变量关系"分析的神器**，
比手写 groupby 更直观。


## 分析 5：性别差异（整体）


In [ ]:
by_sex = penguins.groupby('sex', observed=True)['body_mass_weight_g'].agg(['count', 'mean', 'std']).round(1)
print('整体性别差异：')
print(by_sex)

diff = by_sex['mean'].max() - by_sex['mean'].min()
print(f'\n两性平均体重相差约 {diff:.0f} 克')


---
# 综合结论

把上面的分析汇总成"人话"：


In [ ]:
print('=' * 70)
print('企鹅数据分析结论')
print('=' * 70)
print(f'''
【数据概况】
  · 清洗后有效样本：{len(penguins)} 只
  · 覆盖 {penguins['species'].nunique()} 个种类、{penguins['island'].nunique()} 个岛屿
  · 平均体重：{penguins['body_mass_weight_g'].mean():.0f} 克

【主要发现】
  1. 体型最大的是 {by_species.index[0]}（平均 {by_species['平均体重'].iloc[0]:.0f} 克）
     体型最小的是 {by_species.index[-1]}（平均 {by_species['平均体重'].iloc[-1]:.0f} 克）
  2. 种类和岛屿高度绑定：
     {penguins.groupby('species', observed=True)['island'].agg(lambda x: x.mode()[0]).to_dict()}
  3. 性别也是重要影响因素：{by_sex.index[0]} 平均比 {by_sex.index[1]} 重约 {abs(diff):.0f} 克
  4. 构造的特征 `bill_ratio`（喙长/喙深）比单独的喙长或喙深更有区分度

【方法论提示】
  · 做分组分析时，平均值和标准差要一起看
  · 分类变量之间的"交叉表"能快速发现隐藏的关联
''')


---
## 常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 拿到数据直接分析 | 结论可能建立在脏数据上 | 先 `info()` + `describe()` |
| `value_counts()` 看不到缺失 | 以为没有缺失 | 加 `dropna=False` |
| 随意 `dropna()` | 删太多样本 | 先算缺失比例再决定 |
| `pd.cut` 分箱后分组，出现空组 | 结果里有多余行 | category 列分组加 `observed=True` |
| 只看平均值 | 忽略波动 | 标准差一起看 |
| 用原始列做分析 | 区分度不够 | **构造比例、差值、分箱等新特征** |
| `groupby` 后忘了 `reset_index` | 分组列跑到索引里 | 需要摊平就加 |
| 字符串列不做类型规范 | 占内存、分组慢 | 转 `category` |

---

## 动手练习

**练习 1**：读入 `Penguins.csv`，打印 `shape`、`info()`、以及缺失值统计，说出哪几列有缺失、各缺多少。

**练习 2**：分别用"删行"和"用中位数填充"两种方式处理缺失值，比较最终行数差多少。

**练习 3**：构造一个新特征 `flipper_ratio = flipper_length_mm / body_mass_weight_g`，
并比较三种企鹅在这个特征上的平均值。

**练习 4**：做出"岛屿 × 性别"的交叉表，看哪个组合样本最多、哪个最少。

**练习 5**：找出"喙最长"和"喙最短"的那只企鹅的所有信息。

**练习 6**：把体重分成 `轻/中/重` 三档后，看看三档里各有多少只、分别属于哪些种类。

---

**答案思路（先自己做）**

```python
# 1
p = pd.read_csv('Penguins.csv')
print(p.shape)
p.info()
print(p.isnull().sum())

# 2
print('删行：', len(p.dropna()))
print('填充：', len(p.assign(body_mass_weight_g=p['body_mass_weight_g'].fillna(p['body_mass_weight_g'].median()))))

# 3
p2 = p.dropna().copy()
p2['flipper_ratio'] = p2['flipper_length_mm'] / p2['body_mass_weight_g']
print(p2.groupby('species', observed=True)['flipper_ratio'].mean())

# 4
print(pd.crosstab(p['island'], p['sex']))

# 5
print(p.loc[p['bill_length_mm'].idxmax()])
print(p.loc[p['bill_length_mm'].idxmin()])
```
